In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer

link_df = "https://raw.githubusercontent.com/Ptit-Lulu/Projet2_Cinema_Creuse/donnees_df/df_final_art_acteur.csv" 

df = pd.read_csv(link_df,sep=';')

In [ ]:
#df = df.drop(columns="Unnamed: 21")
df

,tconst,title,startYear,runtimeMinutes,spoken_languages,primaryName,primaryProfession,knownForTitles,averageRating,popularity,...,overview,poster_path,video,budget,revenue,genre_1,genre_2,genre_3,Art_Essai,acteur_principal_
0,tt0035423,Kate et Léopold,2001,118,"['en', 'fr', 'it']",James Mangold,"producer,director,writer","tt3315342,tt11563598,tt1950186,tt0358273",6.4,15.770,...,When her scientist ex-boyfriend discovers a po...,/mUvikzKJJSg9khrVdxK8kg3TMHA.jpg,False,48000000,76019048,Comedy,Fantasy,Romance,No,Meg Ryan
1,tt0036606,Les Coeurs captifs,1983,118,"['en', 'it']",Michael Radford,"director,writer,producer","tt0110877,tt0087803,tt0379889,tt2113659",6.4,1.400,...,Set in 1943 in Scotland during World War II. J...,/anoPMnxdrL4B7EMZZA5tQCmod65.jpg,False,0,0,Drama,War,NaN,No,Phyllis Logan
2,tt0052205,Les folles nuits de Sammy Lee,1963,107,['en'],Ken Hughes,"writer,director,producer","tt0062803,tt0082812,tt0054403,tt0061452",7.1,1.960,...,The compère of a seedy strip club struggles to...,/4Trt44jLU4rTRYg7JK8dbVdXNfe.jpg,False,0,0,Drama,NaN,NaN,No,Anthony Newley
3,tt0052376,Héros de guerre,1961,81,['en'],Burt Topper,"producer,director,writer","tt0065815,tt0074377,tt0053332,tt0052739",5.3,1.400,...,"During the Korean War, a glory-hunting sergean...",/4QyQdnsLNzfGeN1Gzu6xMa7EUz2.jpg,False,0,0,Drama,War,NaN,No,Tony Russel
4,tt0052607,La bataille des sexes,1960,84,['en'],Charles Crichton,"director,editor,writer","tt0095159,tt0037635,tt0045201,tt0044829",6.6,3.273,...,Angela Barrows is a man-eating business woman ...,/i3b1RowQT7IqUyU99xmA3CaQAOt.jpg,False,0,0,Comedy,Crime,NaN,No,Peter Sellers
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23462,tt9894660,Baby Done,2020,91,['en'],Curtis Vowell,"director,actor","tt3074758,tt3850590,tt15333702,tt11646832",6.0,5.750,...,Wannabe-adventurer Zoe freaks out when she fal...,/wHNFvpNTQzRKM9OuCUg24jaxheM.jpg,False,0,0,Comedy,NaN,NaN,No,Rose Matafeo
23463,tt9896916,Le Voyage du Pèlerin,2019,108,['en'],Robert Fernandez,"writer,director,producer","tt7907958,tt32611415,tt22774658,tt2112977",6.5,40.086,...,"An epic journey, faithfully adapted to modern-...",/vtfsNxAsDHElFvYHUc9Khwqg17Y.jpg,False,0,0,Adventure,Animation,Family,No,David Thorpe
23464,tt9898858,Coffee & Kareem,2020,88,['en'],Michael Dowse,"director,producer,writer","tt0388139,tt1486834,tt7734218,tt1456635",5.2,12.953,...,A Detroit cop reluctantly teams with his girlf...,/jFzPMOJrjZfwCxllm3IIEKN7ceF.jpg,False,0,0,Action,Comedy,Crime,No,Ed Helms
23465,tt9902160,Herself,2020,97,['en'],Phyllida Lloyd,"director,miscellaneous,producer","tt0795421,tt1007029,tt9902160,tt6911608",7.0,4.818,...,Struggling to provide her daughters with a saf...,/qTCdyHTibEl90w6rDxTTHTX9k3O.jpg,False,0,0,Drama,NaN,NaN,No,Molly McCann


: 

In [ ]:
df.shape

(23467, 21)

: 

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23467 entries, 0 to 23466
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   tconst             23467 non-null  object 
 1   title              23467 non-null  object 
 2   startYear          23467 non-null  int64  
 3   runtimeMinutes     23467 non-null  int64  
 4   spoken_languages   23467 non-null  object 
 5   primaryName        23467 non-null  object 
 6   primaryProfession  23467 non-null  object 
 7   knownForTitles     23467 non-null  object 
 8   averageRating      23467 non-null  float64
 9   popularity         23467 non-null  float64
 10  tagline            23467 non-null  object 
 11  overview           23467 non-null  object 
 12  poster_path        23467 non-null  object 
 13  video              23467 non-null  bool   
 14  budget             23467 non-null  int64  
 15  revenue            23467 non-null  int64  
 16  genre_1            234

: 

In [ ]:
import nltk
import os

venv_nltk_path = r"J:\Projet GIT\Projet2_Cinema_Creuse\.venv\nltk_data"
nltk.data.path.insert(0, venv_nltk_path)
os.environ["NLTK_DATA"] = venv_nltk_path

# Télécharger les stopwords dans ce dossier
nltk.download('stopwords', download_dir=venv_nltk_path)

[nltk_data] Downloading package stopwords to J:\Projet
[nltk_data]     GIT\Projet2_Cinema_Creuse\.venv\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

: 

In [ ]:
import re
import nltk
from textblob import TextBlob as tb

def clean_fr(phrase):
    # Nettoyer ponctuation
    phrase = re.sub(r"[.,'’`;:!?]+", ' ', phrase)
    
    # Analyse TextBlob
    blob = tb(phrase.lower())
    
    # Lemmatisation
    tokens = [w.lemmatize() for w in blob.words]
    
    # Filtrer stopwords français
    stop_words = set(nltk.corpus.stopwords.words("french"))
    tokens = [t for t in tokens if t not in stop_words]
    
    return " ".join(tokens)

def clean_en(phrase):
    # Nettoyer ponctuation
    phrase = re.sub(r"[.,'’`;:!?]+", ' ', phrase)
    
    # Analyse TextBlob
    blob = tb(phrase.lower())
    
    # Lemmatisation
    tokens = [w.lemmatize() for w in blob.words]
    
    # Filtrer stopwords français
    stop_words = set(nltk.corpus.stopwords.words("english"))
    tokens = [t for t in tokens if t not in stop_words]
    
    return " ".join(tokens)

: 

In [ ]:
from nltk.corpus import stopwords
stop_fr = stopwords.words("french")

: 

: 

: 

In [ ]:
X = df[["primaryName", "overview", "tagline","runtimeMinutes", "startYear","genre_1", "genre_2", "genre_3", "acteur_principal_"]]

for col in ['tagline', 'overview', "primaryName", "acteur_principal_"]:
    X[col] = X[col].replace("Inconnu", "")

X["primaryName"] = X["primaryName"].apply(lambda x : x.replace(" ", ""))
X["acteur_principal_"] = X["acteur_principal_"].apply(lambda x : x.replace(" ", ""))

X['tagline'] = X['tagline'].apply(clean_en)
X['overview'] = X['overview'].apply(clean_en)

nfeature = ["startYear", "runtimeMinutes"]
cfeature = ["primaryName","genre_1", "genre_2", "genre_3" , "acteur_principal_"]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), nfeature),
        ('cat', OneHotEncoder(handle_unknown="ignore"), cfeature),
        ('text_tagline', TfidfVectorizer(max_features=5000, stop_words= "english"), 'tagline'),
        ('text_overview', TfidfVectorizer(max_features=5000, stop_words= "english"), 'overview')
    ])

        

C:\Users\berna\AppData\Local\Temp\ipykernel_3404\1173219957.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = X[col].replace("Inconnu", "")
C:\Users\berna\AppData\Local\Temp\ipykernel_3404\1173219957.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X["primaryName"] = X["primaryName"].apply(lambda x : x.replace(" ", ""))
C:\Users\berna\AppData\Local\Temp\ipykernel_3404\1173219957.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row

: 

: 

: 

In [ ]:
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),                                  # Étape 1 : preprocessing des valeur num (scaler) et valeur cat (encoder)(je ne les scale pas), tfidf sur les textes
    ('model', NearestNeighbors(n_neighbors=5, metric='cosine'))   # Étape 2 : On donne les données propres au modèle
])
pipeline.fit(X)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transforme

: 

: 

: 

In [ ]:
idx_film = df.index[df["title"] == "Pulp Fiction"][0]
donnees_film = X.loc[[idx_film]]

film_transformed = pipeline.named_steps['preprocessor'].transform(donnees_film)
distances, indices = pipeline.named_steps['model'].kneighbors(film_transformed, n_neighbors=11)
voisin_indices = indices[0][1:]  # à partir du 1er élément

voisins = df.iloc[voisin_indices]
print(voisins[["title", "genre_1", "genre_2", "genre_3", "startYear", "Art_Essai", "acteur_principal_"]])

                       title genre_1 genre_2 genre_3  startYear Art_Essai  \
2882   Le Parrain, 2ᵉ partie   Crime   Drama     NaN       1974        No   
7498                  Casino   Crime   Drama     NaN       1995       Yes   
4571                Scarface   Crime   Drama     NaN       1983       Yes   
8512          Secret défense   Crime   Drama     NaN       1998        No   
2480              Le Parrain   Crime   Drama     NaN       1972        No   
10443               Dogville   Crime   Drama     NaN       2003       Yes   
4217   Le prince de New York   Crime   Drama     NaN       1981        No   
9820       Gangs of New York   Crime   Drama     NaN       2002       Yes   
6200   Le Parrain, 3e partie   Crime   Drama     NaN       1990        No   
12709       Le libre arbitre   Crime   Drama     NaN       2006        No   

       acteur_principal_  
2882           Al Pacino  
7498      Robert De Niro  
4571           Al Pacino  
8512   Sandrine Bonnaire  
2480       Marlon

: 

: 

: 

In [ ]:
old_rec = list(df.iloc[indices[0][1:]]["title"])  # stocker ici
print("OLD REC =", old_rec)

OLD REC = ['Le Parrain, 2ᵉ partie', 'Casino', 'Scarface', 'Secret défense', 'Le Parrain', 'Dogville', 'Le prince de New York', 'Gangs of New York', 'Le Parrain, 3e partie', 'Le libre arbitre']


: 

: 

: 

In [ ]:
new_rec = list(df.iloc[indices[0][1:]]["title"])
print("NEW REC =", new_rec)

NEW REC = ['Le Parrain, 2ᵉ partie', 'Casino', 'Scarface', 'Secret défense', 'Le Parrain', 'Dogville', 'Le prince de New York', 'Gangs of New York', 'Le Parrain, 3e partie', 'Le libre arbitre']


: 

: 

: 

In [ ]:


old_set = set(old_rec)
new_set = set(new_rec)

jaccard = len(old_set & new_set) / len(old_set | new_set)
print(jaccard)

def jaccard_score(rec1, rec2):
    set1, set2 = set(rec1), set(rec2)
    return len(set1 & set2) / len(set1 | set2)

score = jaccard_score(old_rec, new_rec)
print("Jaccard :", score)

1.0
Jaccard : 1.0


: 

: 

: 

['knn_model.joblib']

: 

: 

: 